# Solar Filament Segmentation with a Custom U-Net

Train a custom PyTorch U-Net on 512 x 512 grayscale tiles built from the COCO polygon annotations. The model predicts a binary filament mask. At inference time, full-resolution test images are tiled, predicted, and stitched back together. Connected components are encoded as one competition RLE row per filament.


In [ ]:
%pip install -q pycocotools scikit-learn


In [ ]:
# Imports and dataset paths
import os
import csv
import gc
import json
import random
import sys
import time
import warnings
from collections import Counter, defaultdict, deque
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image

# This notebook is configured to require CUDA for model training and inference.
if not torch.cuda.is_available():
    raise RuntimeError(
        "PyTorch cannot access CUDA in this notebook kernel.\n"
        f"Python interpreter: {sys.executable}\n"
        f"PyTorch version: {torch.__version__}\n"
        f"PyTorch CUDA build: {torch.version.cuda!r} "
        "(None means this is a CPU-only PyTorch build).\n"
        "Install a CUDA-enabled PyTorch build into this same Python environment, "
        "select the project .venv kernel, then restart the kernel."
    )
GPU_DEVICE = torch.device("cuda:0")
torch.cuda.set_device(GPU_DEVICE)
print(f"CUDA GPU: {torch.cuda.get_device_name(GPU_DEVICE)}")
print(f"CUDA devices visible: {torch.cuda.device_count()}")
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from pycocotools import mask as mask_utils

warnings.filterwarnings("ignore", category=FutureWarning)

# Set this explicitly only if automatic discovery does not find the dataset.
DATA_ROOT_OVERRIDE = None
DATASET_FOLDER_NAME = "MAGFilo_1.0_Kaggle_2026"
ANNOTATION_FILENAME = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"


def is_dataset_root(path):
    path = Path(path)
    return (
        (path / "train" / "train_images").is_dir()
        and (path / "train" / ANNOTATION_FILENAME).is_file()
        and (path / "test" / "test_images").is_dir()
    )


def find_dataset_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base, base / DATASET_FOLDER_NAME):
            if is_dataset_root(candidate):
                return candidate.resolve()
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        for mount in kaggle_input.iterdir():
            for candidate in (mount, mount / DATASET_FOLDER_NAME):
                if is_dataset_root(candidate):
                    return candidate.resolve()
    raise FileNotFoundError(
        "Could not find the dataset root with train/train_images, annotations, and test/test_images. "
        "Set DATA_ROOT_OVERRIDE to that folder."
    )


DATA_ROOT = (
    Path(DATA_ROOT_OVERRIDE).expanduser().resolve()
    if DATA_ROOT_OVERRIDE
    else find_dataset_root()
)
TRAIN_DIR = DATA_ROOT / "train"
TRAIN_IMAGES_DIR = TRAIN_DIR / "train_images"
TEST_IMAGES_DIR = DATA_ROOT / "test" / "test_images"
ANNOTATION_PATH = TRAIN_DIR / ANNOTATION_FILENAME
TILE_SIZE = 512
IMAGE_WIDTH = 2048
IMAGE_HEIGHT = 2048
TOTAL_PIXELS = IMAGE_WIDTH * IMAGE_HEIGHT

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
sns.set_theme(style="whitegrid", palette="muted", context="notebook")

print(f"Data root: {DATA_ROOT}")
for required_path in (TRAIN_IMAGES_DIR, TEST_IMAGES_DIR, ANNOTATION_PATH):
    if not required_path.exists():
        raise FileNotFoundError(f"Required dataset path not found: {required_path}")

## Small EDA for data counts and visualization


In [ ]:
with open(ANNOTATION_PATH, "r") as f:
    annotations = json.load(f)

annotations["categories"]

In [ ]:
train_image_files = len(os.listdir(TRAIN_IMAGES_DIR))
print(f"Number of training images: {train_image_files}")

test_image_files = len(os.listdir(TEST_IMAGES_DIR))
print(f"Number of test images: {test_image_files}")

annoted_images = len(annotations["images"])
print(f"Number of annotated images: {annoted_images}")

total_annotations = len(annotations["annotations"])
print(f"Number of total annotations: {total_annotations}")

## See annotations for one image with diffrent annotator


In [ ]:
# Map filenames to their corresponding duplicate image ids
name_to_image_ids = defaultdict(list)
for image in annotations["images"]:
    name_to_image_ids[image["file_name"]].append(image["id"])

image_annotator_counts = {
    file_name: len(ids) for file_name, ids in name_to_image_ids.items()
}

distribution = Counter(image_annotator_counts.values())

# Plot 1: Image Count vs. Annotator Count
annotator_counts = sorted(distribution.keys())
image_counts = [distribution[c] for c in annotator_counts]

plt.figure(figsize=(10, 5))
bars = plt.bar(
    annotator_counts, image_counts, color="#4A90E2", edgecolor="black", width=0.6
)

plt.title("Distribution of Images by Annotator Count", fontsize=14, pad=15)
plt.xlabel("Number of Annotators Per Image", fontsize=12)
plt.ylabel("Number of Unique Images", fontsize=12)
plt.xticks(annotator_counts)
plt.grid(axis="y", linestyle="--", alpha=0.5)

# Add value labels on top of bars
for bar in bars:
    yval = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        yval + (yval * 0.01),
        f"{int(yval)}",
        ha="center",
        va="bottom",
        fontsize=10,
        weight="bold",
    )

plt.tight_layout()
plt.show()

In [ ]:
# Plot Example images with multiple annotators
sorted_images = sorted(
    name_to_image_ids.items(), key=lambda item: len(item[1]), reverse=True
)
examples = sorted_images[:6]
colors = ["#FF4B4B", "#1F77B4", "#2CA02C"]

fig, axes = plt.subplots(3, 2, figsize=(20, 20))
axes = axes.flatten()

for i, (file_name, ids) in enumerate(examples):
    ax = axes[i]

    # Load and display the image
    img_path = f"{TRAIN_IMAGES_DIR}/{file_name}"
    image = np.array(Image.open(img_path).convert("RGB"))
    ax.imshow(image, cmap="gray")
    legend_labels = []

    for idx, img_id in enumerate(ids):
        anns = [a for a in annotations["annotations"] if a["image_id"] == img_id]
        color = colors[idx % len(colors)]

        # Plot each polygon for this specific annotator
        for ann in anns:
            for poly in ann["segmentation"]:
                xs = poly[0::2]
                ys = poly[1::2]
                ax.plot(xs + [xs[0]], ys + [ys[0]], color=color, linewidth=1.5)

        # Add to legend tracker if annotations exist
        if anns:
            legend_labels.append(
                plt.Line2D([0], [0], color=color, lw=1.5, label=f"Annotator {idx+1}")
            )

    # Set title, legend, and clean up axes
    ax.set_title(f"{file_name}\n({len(ids)} Annotators)", fontsize=10)
    ax.axis("off")
    if legend_labels:
        ax.legend(handles=legend_labels, loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

# Check out Example Polygon Mask


In [ ]:
def polygons_to_mask(image_id, data):
    img_info = next(img for img in data["images"] if img["id"] == image_id)
    height, width = img_info["height"], img_info["width"]
    mask = np.zeros((height, width), dtype=np.uint8)
    anns = [ann for ann in data["annotations"] if ann["image_id"] == image_id]
    for ann in anns:
        for polygon in ann["segmentation"]:
            points = np.asarray(polygon, dtype=np.float32).reshape((-1, 2))
            if len(points) >= 3:
                cv2.fillPoly(mask, [np.round(points).astype(np.int32)], color=1)
    return mask, img_info


def plot_image_mask_overlay(image, mask, img_info, images_dir):
    image_path = Path(images_dir) / img_info["file_name"]
    image = np.asarray(Image.open(image_path).convert("RGB"))
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(image)
    axes[0].set_title("Image")
    axes[1].imshow(mask, cmap="gray")
    axes[1].set_title("Union of annotated polygons")
    axes[2].imshow(image)
    axes[2].imshow(mask, cmap="Reds", alpha=0.25)
    axes[2].set_title("Overlay")
    for axis in axes:
        axis.axis("off")
    plt.tight_layout()
    plt.show()


sample_index = min(200, len(annotations["images"]) - 1)
image_rnd_id = annotations["images"][sample_index]["id"]
mask, info = polygons_to_mask(image_id=image_rnd_id, data=annotations)
plot_image_mask_overlay(
    image=None, mask=mask, img_info=info, images_dir=TRAIN_IMAGES_DIR
)

# Pre-processing and Data Augmentation


## Split the data into train and validation sets

Images with multiple annotators stay in training. Validation uses only single-annotator images. The split is measured over unique source files so one photograph cannot appear on both sides.


In [ ]:
SPLIT_RATIO = 0.2
RANDOM_SEED = 10


def seed_everything(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(RANDOM_SEED)
print(f"Global seed set to {RANDOM_SEED}")
image_ids = [img["id"] for img in annotations["images"]]
image_annotator_counts = {
    img["id"]: len(name_to_image_ids[img["file_name"]]) for img in annotations["images"]
}
single_annotator_ids = [i for i in image_ids if image_annotator_counts[i] == 1]
multi_annotator_ids = [i for i in image_ids if image_annotator_counts[i] > 1]

val_target_size = max(1, round(len(name_to_image_ids) * SPLIT_RATIO))
if len(single_annotator_ids) < 2:
    raise ValueError("Need at least two single-annotator records for validation.")
val_target_size = min(val_target_size, len(single_annotator_ids) - 1)
train_single_ids, val_image_ids = train_test_split(
    single_annotator_ids,
    test_size=val_target_size,
    random_state=RANDOM_SEED,
    shuffle=True,
)
train_image_ids = train_single_ids + multi_annotator_ids
file_by_id = {img["id"]: img["file_name"] for img in annotations["images"]}
train_files = {file_by_id[i] for i in train_image_ids}
val_files = {file_by_id[i] for i in val_image_ids}
if train_files & val_files:
    raise RuntimeError("Data leakage detected: a source file occurs in both splits.")

print(f"Unique source files: {len(name_to_image_ids):,}")
print(
    f"Training annotation records: {len(train_image_ids):,}; source files: {len(train_files):,}"
)
print(
    f"Validation annotation records: {len(val_image_ids):,}; source files: {len(val_files):,}"
)
print(f"Multi-annotator records kept in training: {len(multi_annotator_ids):,}")

## Build Data-set

- Tile images from 2048x2048 to 512x512 for training and validation.


In [ ]:
def tile_image_and_mask(image_id, data, image_dir, split_name):
    mask, img_info = polygons_to_mask(image_id, data)
    image_path = Path(image_dir) / img_info["file_name"]
    image = np.asarray(Image.open(image_path).convert("L"))
    height, width = mask.shape
    if image.shape != mask.shape:
        raise ValueError(f"Image/mask dimensions differ for {img_info['file_name']}")
    if height % TILE_SIZE or width % TILE_SIZE:
        raise ValueError(f"Image dimensions must be divisible by TILE_SIZE={TILE_SIZE}")

    tiles = []
    for row in range(height // TILE_SIZE):
        for col in range(width // TILE_SIZE):
            y0, x0 = row * TILE_SIZE, col * TILE_SIZE
            image_tile = image[y0 : y0 + TILE_SIZE, x0 : x0 + TILE_SIZE]
            mask_tile = mask[y0 : y0 + TILE_SIZE, x0 : x0 + TILE_SIZE]
            positive_pixels = int(mask_tile.sum())
            tiles.append(
                {
                    "source_image_id": image_id,
                    "source_file_name": img_info["file_name"],
                    "row": row,
                    "col": col,
                    "split": split_name,
                    "tile_name": f"{image_id}_r{row}_c{col}.png",
                    "has_filament": positive_pixels > 0,
                    "positive_pixels": positive_pixels,
                    "image_tile": image_tile,
                    "mask_tile": mask_tile,
                    "orig_height": height,
                    "orig_width": width,
                }
            )
    return tiles


def reconstruct_from_tiles(tiles, tile_type="mask_tile", tile_size=TILE_SIZE):
    if not tiles:
        raise ValueError("The tiles list is empty.")
    height, width = int(tiles[0]["orig_height"]), int(tiles[0]["orig_width"])
    canvas = np.zeros((height, width), dtype=tiles[0][tile_type].dtype)
    for tile in tiles:
        y0, x0 = int(tile["row"]) * tile_size, int(tile["col"]) * tile_size
        canvas[y0 : y0 + tile_size, x0 : x0 + tile_size] = tile[tile_type]
    return canvas

### Validate the tiling process by visualizing a few tiles and their corresponding masks.


In [ ]:
# 1. Tile a specific validation image into 16 patches
validation_visual_id = val_image_ids[0]
validation_tiles = tile_image_and_mask(
    image_id=validation_visual_id,
    data=annotations,
    image_dir=TRAIN_IMAGES_DIR,
    split_name="val",
)

# 2. Reconstruct the full layout
full_reconstructed_mask = reconstruct_from_tiles(
    validation_tiles, tile_type="mask_tile"
)

# Visualize the original image, 16 tiles, and reconstructed image with mask overlays
original_info = next(
    img for img in annotations["images"] if img["id"] == validation_visual_id
)
original_path = TRAIN_IMAGES_DIR / original_info["file_name"]
original_image = np.array(Image.open(original_path).convert("L"))
original_mask, _ = polygons_to_mask(validation_visual_id, annotations)
reconstructed_image = reconstruct_from_tiles(validation_tiles, tile_type="image_tile")
reconstructed_mask = reconstruct_from_tiles(validation_tiles, tile_type="mask_tile")

fig = plt.figure(figsize=(22, 18), constrained_layout=True)
grid = fig.add_gridspec(4, 2, width_ratios=[1, 1], height_ratios=[1, 1, 1, 1])

# Original image with overlay
ax_original = fig.add_subplot(grid[:2, 0])
ax_original.imshow(original_image, cmap="gray")
ax_original.imshow(original_mask, cmap="Reds", alpha=0.3)
ax_original.set_title("Original Image with Mask Overlay")
ax_original.axis("off")

# Reconstructed image with overlay
ax_reconstructed = fig.add_subplot(grid[2:, 0])
ax_reconstructed.imshow(reconstructed_image, cmap="gray")
ax_reconstructed.imshow(reconstructed_mask, cmap="Reds", alpha=0.3)
ax_reconstructed.set_title("Reconstructed Image with Mask Overlay")
ax_reconstructed.axis("off")

# Sixteen tiles in their spatial 4x4 arrangement
subgrid = grid[:, 1].subgridspec(4, 4, wspace=0.03, hspace=0.15)
for tile in validation_tiles:
    ax = fig.add_subplot(subgrid[tile["row"], tile["col"]])
    ax.imshow(tile["image_tile"], cmap="gray")
    ax.imshow(tile["mask_tile"], cmap="Reds", alpha=0.3)
    ax.set_title(f"r{tile['row']} c{tile['col']}", fontsize=9)
    ax.axis("off")

fig.suptitle("Solar Filament Tile and Mask Visualization", fontsize=18)
plt.show()

### Create Tiles and Masks for Training and Validation


In [ ]:
from concurrent.futures import ThreadPoolExecutor

TILED_DATASET_DIR = Path.cwd() / "solar_filament_tiles"
NUM_WORKERS = min(8, os.cpu_count() or 4)

for split_name in ("train", "val"):
    for folder in ("images", "masks"):
        (TILED_DATASET_DIR / split_name / folder).mkdir(parents=True, exist_ok=True)


def process_image(image_id, split_name):
    tiles = tile_image_and_mask(
        image_id=image_id,
        data=annotations,
        image_dir=TRAIN_IMAGES_DIR,
        split_name=split_name,
    )
    records = []
    for tile in tiles:
        image_out = TILED_DATASET_DIR / split_name / "images" / tile["tile_name"]
        mask_out = TILED_DATASET_DIR / split_name / "masks" / tile["tile_name"]

        Image.fromarray(tile["image_tile"].astype(np.uint8)).save(image_out)
        mask_data = (tile["mask_tile"] > 0).astype(np.uint8) * 255
        Image.fromarray(mask_data).save(mask_out)

        records.append(
            {
                key: value
                for key, value in tile.items()
                if key not in {"image_tile", "mask_tile"}
            }
        )

    return records


all_tile_records = []

for split_name, split_image_ids in (("train", train_image_ids), ("val", val_image_ids)):
    split_records = []

    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        futures = [
            executor.submit(process_image, image_id, split_name)
            for image_id in split_image_ids
        ]

        for index, future in enumerate(futures, start=1):
            split_records.extend(future.result())

            if index % 50 == 0 or index == len(futures):
                print(f"{split_name}: processed {index:,}/" f"{len(futures):,} images")

    pd.DataFrame(split_records).to_csv(
        TILED_DATASET_DIR / split_name / "metadata.csv",
        index=False,
    )

    all_tile_records.extend(split_records)
    print(f"{split_name}: {len(split_records):,} tiles written")


pd.DataFrame(all_tile_records).to_csv(
    TILED_DATASET_DIR / "metadata.csv",
    index=False,
)

with open(TILED_DATASET_DIR / "dataset_info.json", "w") as f:
    json.dump(
        {
            "tile_size": 512,
            "image_mode": "L",
            "mask_values": {"background": 0, "filament": 255},
            "train_images": len(train_image_ids),
            "validation_images": len(val_image_ids),
            "train_tiles": sum(row["split"] == "train" for row in all_tile_records),
            "validation_tiles": sum(row["split"] == "val" for row in all_tile_records),
        },
        f,
        indent=2,
    )

print(f"Tiled dataset saved to: {TILED_DATASET_DIR}")

## Custom PyTorch U-Net

The network is implemented here with an encoder, bottleneck, skip connections, and decoder. It learns a binary union mask from polygon annotations. Training keeps positive tiles and samples a configurable number of empty tiles to reduce background-only work.


In [ ]:
# Training and output settings
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA became unavailable. Stop and restart this notebook with a GPU accelerator enabled."
    )
DEVICE = torch.device("cuda:0")
torch.cuda.set_device(DEVICE)
BATCH_SIZE = 4
NUM_EPOCHS = 25
EARLY_STOPPING_PATIENCE = 6
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
BASE_CHANNELS = 32
NEGATIVE_TO_POSITIVE_TILE_RATIO = 2
NUM_WORKERS = 0 if os.name == "nt" else min(4, os.cpu_count() or 1)
AMP_ENABLED = True
GPU_NAME = torch.cuda.get_device_name(DEVICE)
GPU_MEMORY_GIB = torch.cuda.get_device_properties(DEVICE).total_memory / (1024**3)
CPU_COUNT = os.cpu_count() or 1
CPU_POSTPROCESS_WORKERS = min(4, max(1, CPU_COUNT - 1))
MAX_PENDING_POSTPROCESS = CPU_POSTPROCESS_WORKERS * 2
cv2.setNumThreads(1)  # avoid nested OpenCV threads in the CPU worker pool

TILED_DATASET_DIR = Path.cwd() / "solar_filament_tiles"
OUTPUT_DIR = Path.cwd() / "unet_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / "custom_unet_best.pth"
HISTORY_PATH = OUTPUT_DIR / "training_history.csv"
SUBMISSION_PATH = Path.cwd() / "solar_filament_unet_submission.csv"

train_metadata_path = TILED_DATASET_DIR / "train" / "metadata.csv"
val_metadata_path = TILED_DATASET_DIR / "val" / "metadata.csv"
for required_path in (train_metadata_path, val_metadata_path):
    if not required_path.is_file():
        raise FileNotFoundError(
            f"Missing tiled metadata: {required_path}. Run the tile creation cell first."
        )

train_metadata = pd.read_csv(train_metadata_path)
val_metadata = pd.read_csv(val_metadata_path)
is_positive = train_metadata["has_filament"].astype(str).str.lower().eq("true")
train_positive = train_metadata[is_positive]
train_negative = train_metadata[~is_positive]
if train_positive.empty:
    raise ValueError("No filament-positive training tiles were created.")
negative_limit = min(
    len(train_negative), len(train_positive) * NEGATIVE_TO_POSITIVE_TILE_RATIO
)
if negative_limit:
    train_negative = train_negative.sample(n=negative_limit, random_state=RANDOM_SEED)
train_metadata = pd.concat([train_positive, train_negative], ignore_index=True)
train_metadata = train_metadata.sample(frac=1, random_state=RANDOM_SEED).reset_index(
    drop=True
)

print(
    f"Device: {DEVICE}; mixed precision: {AMP_ENABLED}; loader workers: {NUM_WORKERS}"
)
print(
    f"CPU post-processing threads: {CPU_POSTPROCESS_WORKERS}; max queued masks: {MAX_PENDING_POSTPROCESS}"
)
print(
    f"Training tiles: {len(train_metadata):,} (positive {len(train_positive):,}, sampled negative {len(train_negative):,})"
)
print(f"Validation tiles: {len(val_metadata):,}")
print(f"Checkpoint: {CHECKPOINT_PATH}")

In [ ]:
class FilamentTileDataset(Dataset):
    def __init__(self, metadata, split, augment=False):
        self.metadata = metadata.reset_index(drop=True)
        self.image_dir = TILED_DATASET_DIR / split / "images"
        self.mask_dir = TILED_DATASET_DIR / split / "masks"
        self.augment = augment

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, index):
        row = self.metadata.iloc[index]
        image_path = self.image_dir / row["tile_name"]
        mask_path = self.mask_dir / row["tile_name"]
        image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if image is None or mask is None:
            raise OSError(f"Could not read tile pair: {image_path}, {mask_path}")
        image = image.astype(np.float32) / 255.0
        mask = (mask > 127).astype(np.float32)
        if self.augment:
            if random.random() < 0.5:
                image, mask = np.fliplr(image), np.fliplr(mask)
            if random.random() < 0.5:
                image, mask = np.flipud(image), np.flipud(mask)
            turns = random.randrange(4)
            if turns:
                image, mask = np.rot90(image, turns), np.rot90(mask, turns)
        image = np.ascontiguousarray(image[None, ...], dtype=np.float32)
        mask = np.ascontiguousarray(mask[None, ...], dtype=np.float32)
        return torch.from_numpy(image), torch.from_numpy(mask)


train_dataset = FilamentTileDataset(train_metadata, split="train", augment=True)
val_dataset = FilamentTileDataset(val_metadata, split="val", augment=False)
loader_options = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": DEVICE.type == "cuda",
}
if NUM_WORKERS > 0:
    loader_options.update({"persistent_workers": True, "prefetch_factor": 2})
train_loader = DataLoader(train_dataset, shuffle=True, **loader_options)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_options)

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.layers(x)


class CustomUNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, base_channels=BASE_CHANNELS):
        super().__init__()
        c = base_channels
        self.enc1 = DoubleConv(in_channels, c)
        self.enc2 = DoubleConv(c, c * 2)
        self.enc3 = DoubleConv(c * 2, c * 4)
        self.enc4 = DoubleConv(c * 4, c * 8)
        self.bottleneck = DoubleConv(c * 8, c * 16)
        self.pool = nn.MaxPool2d(2, 2)
        self.up4 = nn.ConvTranspose2d(c * 16, c * 8, 2, 2)
        self.dec4 = DoubleConv(c * 16, c * 8)
        self.up3 = nn.ConvTranspose2d(c * 8, c * 4, 2, 2)
        self.dec3 = DoubleConv(c * 8, c * 4)
        self.up2 = nn.ConvTranspose2d(c * 4, c * 2, 2, 2)
        self.dec2 = DoubleConv(c * 4, c * 2)
        self.up1 = nn.ConvTranspose2d(c * 2, c, 2, 2)
        self.dec1 = DoubleConv(c * 2, c)
        self.output = nn.Conv2d(c, out_channels, 1)

    @staticmethod
    def match_size(x, reference):
        if x.shape[-2:] != reference.shape[-2:]:
            x = F.interpolate(
                x, size=reference.shape[-2:], mode="bilinear", align_corners=False
            )
        return x

    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(self.pool(s1))
        s3 = self.enc3(self.pool(s2))
        s4 = self.enc4(self.pool(s3))
        x = self.bottleneck(self.pool(s4))
        x = self.dec4(torch.cat([self.match_size(self.up4(x), s4), s4], dim=1))
        x = self.dec3(torch.cat([self.match_size(self.up3(x), s3), s3], dim=1))
        x = self.dec2(torch.cat([self.match_size(self.up2(x), s2), s2], dim=1))
        x = self.dec1(torch.cat([self.match_size(self.up1(x), s1), s1], dim=1))
        return self.output(x)


model = CustomUNet().to(DEVICE)
if not next(model.parameters()).is_cuda:
    raise RuntimeError("U-Net parameters are not on CUDA; refusing to train on CPU.")
print(f"Model device: {next(model.parameters()).device}")
print(model)
print(
    f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
)

## Train and select the checkpoint

Exact Panoptic Quality uses thresholded instances and discrete IoU matching, so it is not differentiable. Training uses a smooth PQ surrogate on the binary union masks: soft segmentation quality (IoU) multiplied by soft recognition quality (Dice/F1). This gives Adam a gradient while keeping the objective aligned with PQ. The validation loss selects the checkpoint; the probability threshold is selected later using a hard connected-component PQ proxy on validation union masks.


In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-6
)
scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)


def soft_panoptic_quality_loss(logits, targets, smooth=1.0):
    """Differentiable PQ surrogate for the notebook's binary union masks."""
    probabilities = torch.sigmoid(logits)
    intersection = (probabilities * targets).sum(dim=(1, 2, 3))
    predicted_area = probabilities.sum(dim=(1, 2, 3))
    target_area = targets.sum(dim=(1, 2, 3))
    union = predicted_area + target_area - intersection

    soft_sq = (intersection + smooth) / (union + smooth)
    soft_rq = (2.0 * intersection + smooth) / (predicted_area + target_area + smooth)
    soft_pq = soft_sq * soft_rq

    # Empty-mask tiles have no matched objects; penalize their predicted foreground smoothly.
    pixel_count = targets.shape[-2] * targets.shape[-1]
    empty_score = torch.exp(-predicted_area / max(1.0, pixel_count * 0.005))
    score = torch.where(target_area > 0, soft_pq, empty_score)
    return 1.0 - score.mean()


def run_epoch(loader, training):
    if (
        DEVICE.type != "cuda"
        or not torch.cuda.is_available()
        or not next(model.parameters()).is_cuda
    ):
        raise RuntimeError("CUDA is required; model parameters are not on the GPU.")
    model.train(training)
    total_loss = 0.0
    intersection_total = 0
    denominator_total = 0
    sample_count = 0
    total_batches = len(loader)
    phase = "train" if training else "val"
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for batch_index, (images, targets) in enumerate(loader, start=1):
            images = images.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            if not images.is_cuda or not targets.is_cuda:
                raise RuntimeError("A training batch was not transferred to CUDA.")
            if training:
                optimizer.zero_grad(set_to_none=True)
            with torch.autocast(
                device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED
            ):
                logits = model(images)
                loss = soft_panoptic_quality_loss(logits, targets)
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += float(loss.detach().item()) * len(images)
            predictions = torch.sigmoid(logits.detach()) >= 0.5
            target_binary = targets >= 0.5
            intersection_total += (
                torch.logical_and(predictions, target_binary).sum().item()
            )
            denominator_total += predictions.sum().item() + target_binary.sum().item()
            sample_count += len(images)
            if (
                batch_index == 1
                or batch_index % 10 == 0
                or batch_index == total_batches
            ):
                print(
                    f"  {phase}: batch {batch_index:,}/{total_batches:,} | loss {loss.item():.4f}",
                    flush=True,
                )
    mean_loss = total_loss / max(sample_count, 1)
    mean_dice = (2 * intersection_total + 1.0) / (denominator_total + 1.0)
    return mean_loss, mean_dice


history = []
best_val_loss = float("inf")
stale_epochs = 0
training_start = time.perf_counter()
for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.perf_counter()
    print(f"\nEpoch {epoch:02d}/{NUM_EPOCHS} started", flush=True)
    train_loss, train_dice = run_epoch(train_loader, training=True)
    val_loss, val_dice = run_epoch(val_loader, training=False)
    scheduler.step(val_loss)
    learning_rate = optimizer.param_groups[0]["lr"]
    history.append(
        {
            "epoch": epoch,
            "train_soft_pq_loss": train_loss,
            "train_dice_at_0_5": train_dice,
            "val_soft_pq_loss": val_loss,
            "val_dice_at_0_5": val_dice,
            "learning_rate": learning_rate,
        }
    )
    if val_loss < best_val_loss - 1e-5:
        best_val_loss = val_loss
        stale_epochs = 0
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "epoch": epoch,
                "val_soft_pq_loss": val_loss,
                "tile_size": TILE_SIZE,
                "base_channels": BASE_CHANNELS,
            },
            CHECKPOINT_PATH,
        )
        checkpoint_status = "saved best"
    else:
        stale_epochs += 1
        checkpoint_status = f"no improvement ({stale_epochs}/{EARLY_STOPPING_PATIENCE})"
    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | train soft-PQ loss {train_loss:.4f}, Dice {train_dice:.4f} | "
        f"val soft-PQ loss {val_loss:.4f}, Dice {val_dice:.4f} | lr {learning_rate:.2e} | "
        f"{checkpoint_status} | {time.perf_counter() - epoch_start:.1f}s",
        flush=True,
    )
    if stale_epochs >= EARLY_STOPPING_PATIENCE:
        print("Early stopping: validation soft-PQ loss did not improve.")
        break

history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_PATH, index=False)
if not CHECKPOINT_PATH.is_file():
    raise RuntimeError("Training finished without writing a best-model checkpoint.")
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(
    f"Best checkpoint epoch: {checkpoint['epoch']}; validation soft-PQ loss: {checkpoint['val_soft_pq_loss']:.4f}"
)
print(f"Training time: {(time.perf_counter() - training_start) / 60:.1f} minutes")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["epoch"], history_df["train_soft_pq_loss"], label="Train")
axes[0].plot(history_df["epoch"], history_df["val_soft_pq_loss"], label="Validation")
axes[0].set(title="Soft Panoptic Quality loss", xlabel="Epoch", ylabel="Soft-PQ loss")
axes[0].legend()
axes[1].plot(history_df["epoch"], history_df["train_dice_at_0_5"], label="Train")
axes[1].plot(history_df["epoch"], history_df["val_dice_at_0_5"], label="Validation")
axes[1].set(title="Global pixel Dice at threshold 0.5", xlabel="Epoch", ylabel="Dice")
axes[1].legend()
plt.tight_layout()
plt.show()

## Select the threshold by validation Panoptic Quality

For each candidate, threshold validation masks, split them into connected components, match components by IoU, and compute dataset-level connected-component PQ on the binary union masks. This component-PQ proxy is used for threshold selection; the differentiable soft-PQ surrogate is used during training.


In [ ]:
THRESHOLD_CANDIDATES = (0.20, 0.30, 0.40, 0.50, 0.60, 0.70)
PQ_MATCH_IOU = 0.50
MIN_COMPONENT_AREA = 20
threshold_totals = {
    t: {
        "intersection": 0,
        "predicted_pixels": 0,
        "target_pixels": 0,
        "matched_iou": 0.0,
        "tp": 0,
        "fp": 0,
        "fn": 0,
    }
    for t in THRESHOLD_CANDIDATES
}


def connected_components(binary_mask, min_area=MIN_COMPONENT_AREA):
    count, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary_mask.astype(np.uint8), connectivity=8
    )
    return [
        labels == label
        for label in range(1, count)
        if int(stats[label, cv2.CC_STAT_AREA]) >= min_area
    ]


def panoptic_quality_counts(prediction, target, iou_threshold=PQ_MATCH_IOU):
    predicted_instances = connected_components(prediction)
    target_instances = connected_components(target)
    if not predicted_instances and not target_instances:
        return {"matched_iou": 0.0, "tp": 0, "fp": 0, "fn": 0}

    iou_matrix = np.zeros(
        (len(target_instances), len(predicted_instances)), dtype=np.float32
    )
    for gt_index, gt_mask in enumerate(target_instances):
        gt_area = int(gt_mask.sum())
        for pred_index, pred_mask in enumerate(predicted_instances):
            intersection = int(np.logical_and(gt_mask, pred_mask).sum())
            if intersection:
                union = gt_area + int(pred_mask.sum()) - intersection
                iou_matrix[gt_index, pred_index] = intersection / union

    pairs = [
        (float(iou_matrix[g, pred]), g, pred)
        for g in range(len(target_instances))
        for pred in range(len(predicted_instances))
        if iou_matrix[g, pred] > iou_threshold
    ]
    pairs.sort(reverse=True)
    matched_gt, matched_pred, matched_iou = set(), set(), []
    for iou, gt_index, pred_index in pairs:
        if gt_index not in matched_gt and pred_index not in matched_pred:
            matched_gt.add(gt_index)
            matched_pred.add(pred_index)
            matched_iou.append(iou)
    tp = len(matched_iou)
    return {
        "matched_iou": float(sum(matched_iou)),
        "tp": tp,
        "fp": len(predicted_instances) - tp,
        "fn": len(target_instances) - tp,
    }


def _pq_worker(pair):
    return panoptic_quality_counts(pair[0], pair[1])


pq_executor = ThreadPoolExecutor(max_workers=CPU_POSTPROCESS_WORKERS)
model.eval()
with torch.no_grad():
    for images, targets in val_loader:
        images = images.to(DEVICE, non_blocking=True)
        target_binary = targets[:, 0].cpu().numpy() >= 0.5
        with torch.autocast(
            device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED
        ):
            probabilities = torch.sigmoid(model(images))[:, 0].float().cpu().numpy()
        for threshold in THRESHOLD_CANDIDATES:
            predicted_binary = probabilities >= threshold
            totals = threshold_totals[threshold]
            totals["intersection"] += int(
                np.logical_and(predicted_binary, target_binary).sum()
            )
            totals["predicted_pixels"] += int(predicted_binary.sum())
            totals["target_pixels"] += int(target_binary.sum())
            pq_futures = [
                pq_executor.submit(_pq_worker, pair)
                for pair in zip(predicted_binary, target_binary)
            ]
            for future in pq_futures:
                counts = future.result()
                for key in ("matched_iou", "tp", "fp", "fn"):
                    totals[key] += counts[key]

pq_executor.shutdown(wait=True)
threshold_rows = []
for threshold, totals in threshold_totals.items():
    denominator = totals["tp"] + 0.5 * totals["fp"] + 0.5 * totals["fn"]
    pq = totals["matched_iou"] / denominator if denominator else 1.0
    pixel_dice = (2 * totals["intersection"] + 1.0) / (
        totals["predicted_pixels"] + totals["target_pixels"] + 1.0
    )
    threshold_rows.append(
        {
            "threshold": threshold,
            "validation_component_PQ": pq,
            "validation_pixel_Dice": pixel_dice,
            "TP": totals["tp"],
            "FP": totals["fp"],
            "FN": totals["fn"],
        }
    )
threshold_summary = (
    pd.DataFrame(threshold_rows)
    .sort_values(["validation_component_PQ", "validation_pixel_Dice"], ascending=False)
    .reset_index(drop=True)
)
BEST_THRESHOLD = float(threshold_summary.iloc[0]["threshold"])
display(threshold_summary.round(4))
print(f"Selected threshold by validation component-PQ proxy: {BEST_THRESHOLD:.2f}")

In [ ]:
def predict_full_resolution(image, threshold=BEST_THRESHOLD, batch_size=4):
    if image.ndim != 2:
        raise ValueError(f"Expected grayscale image, got shape {image.shape}")
    height, width = image.shape
    pad_height, pad_width = (-height) % TILE_SIZE, (-width) % TILE_SIZE
    if pad_height or pad_width:
        mode = (
            cv2.BORDER_REFLECT_101 if min(height, width) > 1 else cv2.BORDER_REPLICATE
        )
        padded = cv2.copyMakeBorder(image, 0, pad_height, 0, pad_width, mode)
    else:
        padded = image
    padded_height, padded_width = padded.shape
    positions = [
        (y, x)
        for y in range(0, padded_height, TILE_SIZE)
        for x in range(0, padded_width, TILE_SIZE)
    ]
    stitched = np.zeros((padded_height, padded_width), dtype=np.uint8)
    model.eval()
    with torch.no_grad():
        for start in range(0, len(positions), batch_size):
            batch_positions = positions[start : start + batch_size]
            patches = (
                np.stack(
                    [
                        padded[y : y + TILE_SIZE, x : x + TILE_SIZE]
                        for y, x in batch_positions
                    ]
                ).astype(np.float32)
                / 255.0
            )
            inputs = torch.from_numpy(patches[:, None, ...]).to(DEVICE)
            with torch.autocast(
                device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED
            ):
                probabilities = torch.sigmoid(model(inputs)).float().cpu().numpy()[:, 0]
            for (y, x), probability in zip(batch_positions, probabilities):
                stitched[y : y + TILE_SIZE, x : x + TILE_SIZE] = (
                    probability >= threshold
                ).astype(np.uint8)
    return stitched[:height, :width].astype(bool)


# Visualize a held-out validation image at native resolution.
validation_visual_id = val_image_ids[0]
validation_info = next(
    img for img in annotations["images"] if img["id"] == validation_visual_id
)
validation_image = cv2.imread(
    str(TRAIN_IMAGES_DIR / validation_info["file_name"]), cv2.IMREAD_GRAYSCALE
)
validation_target, _ = polygons_to_mask(validation_visual_id, annotations)
validation_prediction = predict_full_resolution(validation_image)
intersection = np.logical_and(validation_prediction, validation_target > 0).sum()
denominator = validation_prediction.sum() + (validation_target > 0).sum()
image_dice = (2 * intersection + 1.0) / (denominator + 1.0)

fig, axes = plt.subplots(1, 3, figsize=(17, 6))
axes[0].imshow(validation_image, cmap="gray")
axes[0].set_title("Held-out image")
axes[1].imshow(validation_image, cmap="gray")
axes[1].imshow(validation_target, cmap="Reds", alpha=0.35)
axes[1].set_title("Ground truth")
axes[2].imshow(validation_image, cmap="gray")
axes[2].imshow(validation_prediction, cmap="winter", alpha=0.4)
axes[2].set_title(f"U-Net prediction (Dice={image_dice:.3f})")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

## Generate the competition submission

Submission formatting follows the YOLO notebook: split the binary prediction into 8-connected components, discard components smaller than 20 pixels, sort by descending area (then top/left position), assign IDs as image-stem plus component number, and encode each component with pycocotools in Fortran order.


In [ ]:
MIN_COMPONENT_AREA = 20
CREATE_SUBMISSION = True


def get_test_image_paths():
    suffixes = {".jpg", ".jpeg", ".png"}
    paths = [
        p
        for p in TEST_IMAGES_DIR.iterdir()
        if p.is_file() and p.suffix.lower() in suffixes
    ]
    return sorted(paths, key=lambda path: path.name)


def split_into_filaments(binary_mask, min_area=MIN_COMPONENT_AREA):
    count, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary_mask.astype(np.uint8), connectivity=8
    )
    components = []
    for label in range(1, count):
        area = int(stats[label, cv2.CC_STAT_AREA])
        if area >= min_area:
            top = int(stats[label, cv2.CC_STAT_TOP])
            left = int(stats[label, cv2.CC_STAT_LEFT])
            components.append((area, top, left, labels == label))
    components.sort(key=lambda item: (-item[0], item[1], item[2]))
    return [item[3] for item in components]


def mask_to_rle(binary_mask):
    encoded = mask_utils.encode(np.asfortranarray(binary_mask.astype(np.uint8)))
    return encoded["counts"].decode("utf-8")


def encode_prediction_rows(path_stem, binary_mask):
    components = split_into_filaments(binary_mask, MIN_COMPONENT_AREA)
    return [
        (f"{path_stem}_{index}", mask_to_rle(component))
        for index, component in enumerate(components, start=1)
    ]


def create_submission(output_path=SUBMISSION_PATH):
    image_paths = get_test_image_paths()
    if not image_paths:
        raise FileNotFoundError(f"No test images found in {TEST_IMAGES_DIR}")
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if DEVICE.type == "cpu":
        # Leave CPU cores available while RLE workers run beside inference.
        torch.set_num_threads(max(1, CPU_COUNT - CPU_POSTPROCESS_WORKERS))

    start_time = time.perf_counter()
    total_rows = 0
    seen_ids = set()
    pending = deque()

    with ThreadPoolExecutor(max_workers=CPU_POSTPROCESS_WORKERS) as executor:
        with output_path.open("w", newline="", encoding="utf-8") as submission_file:
            writer = csv.writer(submission_file)
            writer.writerow(["filament_id", "segmentation_rle"])

            def write_oldest_result():
                nonlocal total_rows
                _, future = pending.popleft()
                for filament_id, rle in future.result():
                    if filament_id in seen_ids:
                        raise ValueError(f"Duplicate filament ID: {filament_id}")
                    seen_ids.add(filament_id)
                    writer.writerow([filament_id, rle])
                    total_rows += 1

            for index, image_path in enumerate(image_paths, start=1):
                image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
                if image is None:
                    raise OSError(f"Could not read test image: {image_path}")
                prediction = predict_full_resolution(image, threshold=BEST_THRESHOLD)
                future = executor.submit(
                    encode_prediction_rows, image_path.stem, prediction
                )
                pending.append((image_path.stem, future))
                del prediction, image

                # Bound queued full-resolution masks while CPU RLE work overlaps inference.
                if len(pending) >= MAX_PENDING_POSTPROCESS:
                    write_oldest_result()
                if index % 20 == 0 or index == len(image_paths):
                    print(
                        f"Inference {index:,}/{len(image_paths):,}; "
                        f"rows written {total_rows:,}; queued CPU tasks {len(pending)}",
                        flush=True,
                    )

            while pending:
                write_oldest_result()

    print(
        f"Submission saved: {output_path}\nTest images: {len(image_paths):,}; "
        f"filament rows: {total_rows:,}; elapsed: {(time.perf_counter() - start_time) / 60:.1f} min"
    )
    return output_path


if CREATE_SUBMISSION:
    create_submission()
else:
    print(f"Set CREATE_SUBMISSION=True to write {SUBMISSION_PATH}")

In [ ]:
# Stream through the CSV and check its schema, IDs, and a decodable sample RLE.
submission_columns = None
submission_row_count = 0
null_row_count = 0
duplicate_id_count = 0
empty_value_count = 0
seen_ids = set()
sample_rle_decodable = False

if CREATE_SUBMISSION:
    with SUBMISSION_PATH.open("r", newline="", encoding="utf-8") as submission_file:
        reader = csv.DictReader(submission_file)
        submission_columns = reader.fieldnames
        for row in reader:
            submission_row_count += 1
            filament_id = row.get("filament_id")
            rle = row.get("segmentation_rle")
            if any(value is None for value in row.values()):
                null_row_count += 1
            if not filament_id or not rle:
                empty_value_count += 1
            if filament_id in seen_ids:
                duplicate_id_count += 1
            elif filament_id:
                seen_ids.add(filament_id)
            if submission_row_count == 1 and rle:
                try:
                    decoded = mask_utils.decode(
                        {
                            "size": [IMAGE_HEIGHT, IMAGE_WIDTH],
                            "counts": rle.encode("utf-8"),
                        }
                    )
                    sample_rle_decodable = decoded.shape == (
                        IMAGE_HEIGHT,
                        IMAGE_WIDTH,
                    ) and bool(decoded.any())
                except Exception:
                    sample_rle_decodable = False

submission_valid = (
    submission_columns == ["filament_id", "segmentation_rle"]
    and null_row_count == 0
    and duplicate_id_count == 0
    and empty_value_count == 0
    and (sample_rle_decodable or submission_row_count == 0)
)
print("--- Submission audit ---")
print(f"Columns: {submission_columns}")
print(f"Rows: {submission_row_count:,}")
print(f"Null rows: {null_row_count:,}")
print(f"Duplicate IDs: {duplicate_id_count:,}")
print(f"Empty IDs/RLE: {empty_value_count:,}")
print(f"Sample RLE decodes to {IMAGE_HEIGHT}x{IMAGE_WIDTH}: {sample_rle_decodable}")
print(
    "[VERIFIED] Submission structure passed."
    if submission_valid
    else "[WARNING] Submission needs inspection."
)

## Outputs

- solar_filament_tiles/ ? generated 512 x 512 image and mask tiles with split metadata.
- unet_outputs/custom_unet_best.pth ? best custom U-Net checkpoint.
- unet_outputs/training_history.csv ? epoch-level loss and Dice history.
- solar_filament_unet_submission.csv ? competition rows in filament_id, segmentation_rle format.

CPU work uses bounded thread pools for tile creation, validation component matching, and submission RLE encoding. Adjust CPU_POSTPROCESS_WORKERS to change submission and validation parallelism. Adjust NUM_EPOCHS, BATCH_SIZE, BASE_CHANNELS, and the negative-tile ratio to fit the available runtime and GPU memory.
